In [1]:
import os
import warnings
import logging
warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.ERROR)

import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras import layers
from tensorflow.keras.utils import register_keras_serializable
from tensorflow.keras.layers import (
    Conv2D, Dense, Layer,
    GlobalAveragePooling2D, GlobalMaxPooling2D,
    Add, Reshape, Multiply, Activation, Concatenate
)
from tensorflow.keras import layers, initializers

import numpy as np
import time
import psutil
import platform

In [2]:
# suppress verbose TF/Keras logs
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
logging.getLogger('tensorflow').setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

# ================================================================
#  HELPERS
# ================================================================
def _dtype_itemsize(dtype):
    """Works for both string dtypes (Keras 3) and numpy dtypes."""
    if isinstance(dtype, str):
        return np.dtype(dtype).itemsize
    return dtype.itemsize


def count_parameters(model):
    trainable     = int(sum(np.prod(w.shape) for w in model.trainable_weights))
    non_trainable = int(sum(np.prod(w.shape) for w in model.non_trainable_weights))
    return {
        'trainable'    : trainable,
        'non_trainable': non_trainable,
        'total'        : trainable + non_trainable,
        'trainable_M'  : round(trainable / 1e6, 4),
        'total_M'      : round((trainable + non_trainable) / 1e6, 4),
    }


def get_flops(model, input_shape=(224, 224, 3)):
    """
    Compute FLOPs silently — suppresses the verbose profile report.
    Returns raw FLOPs integer or None on failure.
    """
    try:
        from tensorflow.python.framework.convert_to_constants import \
            convert_variables_to_constants_v2

        dummy  = tf.ones((1, *input_shape))
        cf     = tf.function(model).get_concrete_function(dummy)
        frozen = convert_variables_to_constants_v2(cf)
        graph  = frozen.graph

        # redirect stdout to suppress the profile ASCII table
        import io, sys
        old_stdout = sys.stdout
        sys.stdout = io.StringIO()

        with graph.as_default():
            run_meta = tf.compat.v1.RunMetadata()
            opts = (tf.compat.v1.profiler.ProfileOptionBuilder
                    .float_operation())
            opts['output'] = 'none'           # ← suppresses stdout dump
            prof = tf.compat.v1.profiler.profile(
                graph=graph, run_meta=run_meta,
                cmd='op', options=opts)

        sys.stdout = old_stdout
        return prof.total_float_ops

    except Exception as e:
        sys.stdout = old_stdout if 'old_stdout' in dir() else sys.stdout
        print(f"    ⚠️  FLOPs computation failed: {e}")
        return None


def measure_memory(model, input_shape=(224, 224, 3)):
    process    = psutil.Process(os.getpid())
    ram_before = process.memory_info().rss / 1024 ** 2

    dummy = np.random.rand(1, *input_shape).astype(np.float32)
    _     = model.predict(dummy, verbose=0)

    ram_after = process.memory_info().rss / 1024 ** 2

    # ── fix: handle string dtype (Keras 3) ──────────────────
    param_bytes = sum(
        np.prod(w.shape) * _dtype_itemsize(w.dtype)
        for w in model.weights
    )
    weight_mb = param_bytes / 1024 ** 2

    # disk size
    tmp_path = '/tmp/_model_deploy_check.keras'
    model.save(tmp_path)
    disk_mb = os.path.getsize(tmp_path) / 1024 ** 2
    os.remove(tmp_path)

    return {
        'weight_size_mb': round(weight_mb, 3),
        'disk_size_mb'  : round(disk_mb, 3),
        'ram_before_mb' : round(ram_before, 2),
        'ram_after_mb'  : round(ram_after, 2),
        'ram_delta_mb'  : round(ram_after - ram_before, 2),
    }


def measure_latency(model, input_shape=(224, 224, 3),
                    batch_size=1, n_warmup=20, n_runs=200):
    dummy = np.random.rand(batch_size, *input_shape).astype(np.float32)

    for _ in range(n_warmup):
        model.predict(dummy, verbose=0)

    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        model.predict(dummy, verbose=0)
        times.append((time.perf_counter() - t0) * 1000)   # ms

    times        = np.array(times)
    per_image_ms = times / batch_size
    return {
        'batch_size'       : batch_size,
        'mean_ms'          : float(np.mean(per_image_ms)),
        'std_ms'           : float(np.std(per_image_ms)),
        'median_ms'        : float(np.median(per_image_ms)),
        'p95_ms'           : float(np.percentile(per_image_ms, 95)),
        'p99_ms'           : float(np.percentile(per_image_ms, 99)),
        'throughput_img_s' : float(batch_size * 1000 / np.mean(times)),
    }


def get_hardware_info():
    gpus = tf.config.list_physical_devices('GPU')
    info = {
        'platform'    : platform.platform(),
        'processor'   : platform.processor(),
        'python'      : platform.python_version(),
        'tensorflow'  : tf.__version__,
        'cpu_cores'   : psutil.cpu_count(logical=False),
        'cpu_threads' : psutil.cpu_count(logical=True),
        'ram_total_gb': round(psutil.virtual_memory().total / 1024**3, 2),
        'gpu_devices' : [g.name for g in gpus] if gpus else ['CPU only'],
    }
    if gpus:
        try:
            det = tf.config.experimental.get_device_details(gpus[0])
            info['gpu_name'] = det.get('device_name', 'N/A')
        except Exception:
            pass
    return info


# ================================================================
#  MAIN
# ================================================================
def full_deployment_profile(model, input_shape=(224, 224, 3),
                             batch_sizes=(1, 8, 16, 32)):

    SEP = "=" * 62

    print(f"\n{SEP}")
    print("  DEPLOYMENT METRICS PROFILE")
    print(SEP)

    # ── hardware ────────────────────────────────────────────
    hw = get_hardware_info()
    print("\n  [Hardware]")
    for k, v in hw.items():
        print(f"    {k:<20}: {v}")

    # ── parameters ──────────────────────────────────────────
    params = count_parameters(model)
    print(f"\n  [Parameters]")
    print(f"    Trainable     : {params['trainable']:>12,}  ({params['trainable_M']} M)")
    print(f"    Non-trainable : {params['non_trainable']:>12,}")
    print(f"    Total         : {params['total']:>12,}  ({params['total_M']} M)")

    # ── FLOPs ───────────────────────────────────────────────
    print(f"\n  [FLOPs]")
    flops_raw = get_flops(model, input_shape)
    if flops_raw:
        print(f"    Total FLOPs   : {flops_raw:>15,}")
        print(f"    MFLOPs        : {flops_raw/1e6:>15.2f}")
        print(f"    GFLOPs        : {flops_raw/1e9:>15.4f}")
    else:
        print("    Could not compute FLOPs.")

    # ── memory ──────────────────────────────────────────────
    print(f"\n  [Memory & Model Size]")
    mem = measure_memory(model, input_shape)
    print(f"    Weight size   : {mem['weight_size_mb']:>8.3f} MB")
    print(f"    Disk size     : {mem['disk_size_mb']:>8.3f} MB")
    print(f"    RAM (before)  : {mem['ram_before_mb']:>8.2f} MB")
    print(f"    RAM (after)   : {mem['ram_after_mb']:>8.2f} MB")
    print(f"    RAM delta     : {mem['ram_delta_mb']:>8.2f} MB")

    # ── latency & throughput ─────────────────────────────────
    print(f"\n  [Latency & Throughput]  ({200} timed runs per batch)")
    print(f"  {'Batch':>6} | {'Mean ms':>9} | {'Std ms':>7} | "
          f"{'P95 ms':>8} | {'P99 ms':>8} | {'img/s':>10}")
    print(f"  {'-'*6}-+-{'-'*9}-+-{'-'*7}-+-{'-'*8}-+-{'-'*8}-+-{'-'*10}")

    latency_results = {}
    for bs in batch_sizes:
        r = measure_latency(model, input_shape, batch_size=bs)
        latency_results[bs] = r
        print(f"  {bs:>6} | {r['mean_ms']:>9.3f} | {r['std_ms']:>7.3f} | "
              f"{r['p95_ms']:>8.3f} | {r['p99_ms']:>8.3f} | "
              f"{r['throughput_img_s']:>10.1f}")

    r1 = latency_results[1]
    print(f"\n  [Single-Image Summary  (batch=1)]")
    print(f"    Mean latency  : {r1['mean_ms']:.3f} ± {r1['std_ms']:.3f} ms")
    print(f"    Median        : {r1['median_ms']:.3f} ms")
    print(f"    95th pct      : {r1['p95_ms']:.3f} ms")
    print(f"    Throughput    : {r1['throughput_img_s']:.1f} images/sec")
    print(f"\n{SEP}\n")

    return dict(hardware=hw, params=params, memory=mem,
                latency=latency_results,
                flops_raw=flops_raw or 'N/A')

In [3]:
# serializable functions for loading the model. These have no work in this file
@tf.keras.utils.register_keras_serializable(package="Custom", name="F1Score")
class F1Score(tf.keras.metrics.Metric):
    """
    Custom Keras metric to compute the F1 Score.
    The F1 score is the harmonic mean of precision and recall.
    """

    def __init__(self, name='f1_score', **kwargs):
        """
        Initializes the F1Score metric. 
        - Uses Keras' Precision and Recall metrics as intermediate steps.
        """
        super().__init__(name=name, **kwargs)
        self.precision = tf.keras.metrics.Precision()  # Precision metric
        self.recall = tf.keras.metrics.Recall()  # Recall metric

    def update_state(self, y_true, y_pred, sample_weight=None):
        """
        Updates the state of the metric.
        - This method is called during training to update precision and recall.
        - The precision and recall states are updated based on true and predicted values.
        """
        self.precision.update_state(y_true, y_pred, sample_weight)
        self.recall.update_state(y_true, y_pred, sample_weight)

    def result(self):
        """
        Computes and returns the F1 score.
        - F1 score is calculated as the harmonic mean of precision and recall.
        - Prevents division by zero by adding a small epsilon value to the denominator.
        """
        p = self.precision.result()  # Get precision value
        r = self.recall.result()  # Get recall value
        return 2 * (p * r) / (p + r + tf.keras.backend.epsilon())  # Calculate F1 score

    def reset_states(self):
        """
        Resets the states of the precision and recall metrics.
        - This is called at the beginning of each epoch.
        """
        self.precision.reset_states()  # Reset precision state
        self.recall.reset_states()  # Reset recall state


######## Main Model which is used in this study of breast cancer###########
@register_keras_serializable(package="Custom", name="KANLayer")
class KANLayer(Layer):
    def __init__(self, units, grid=5, spline_order=3, dropout=0.0, **kwargs):
        """
        KANLayer: Kernel Attention Network Layer (self-explainable)
        - units: output dimension
        - grid: number of spline bins
        - spline_order: reserved for higher-order splines
        - dropout: dropout rate
        """
        super(KANLayer, self).__init__(**kwargs)
        self.units = int(units)
        self.grid = int(grid)
        self.spline_order = int(spline_order)
        self.dropout = float(dropout)

    def build(self, input_shape):
        if len(input_shape) != 2:
            raise ValueError(f"KANLayer expects 2D input (batch, features). Got shape: {input_shape}")

        input_dim = int(input_shape[-1])

        # Linear weights (like Dense)
        self.w = self.add_weight(
            shape=(input_dim, self.units),
            initializer=initializers.HeNormal(),
            trainable=True,
            name="linear_weight"
        )

        # Spline weights: (grid, input_dim, units)
        self.spline_w = self.add_weight(
            shape=(self.grid, input_dim, self.units),
            initializer="glorot_uniform",
            trainable=True,
            name="spline_weight"
        )

        # Bias
        self.b = self.add_weight(
            shape=(self.units,),
            initializer="zeros",
            trainable=True,
            name="bias"
        )

        super(KANLayer, self).build(input_shape)

    def call(self, inputs, training=False):
        x = tf.cast(inputs, tf.float32)

        # --- Linear part ---
        linear_out = tf.linalg.matmul(x, self.w)

        # --- Spline / soft-binning part ---
        x_exp = tf.expand_dims(x, axis=1)  # (B, 1, D)

        # Fixed 0-1 grid (assumes input normalized to 0-1)
        grid_centers = tf.linspace(0.0, 1.0, self.grid)
        grid_centers = tf.reshape(grid_centers, (1, self.grid, 1))  # (1, grid, 1)
        grid_centers = tf.cast(grid_centers, tf.float32)

        # Soft RBF-like basis: (B, grid, D)
        spline_basis = tf.exp(-tf.square(x_exp - grid_centers))

        # einsum: "bgd,gdu->bu" -> (B, units)
        spline_out = tf.einsum("bgd,gdu->bu", spline_basis, self.spline_w)

        out = linear_out + spline_out + self.b

        if self.dropout and training:
            out = tf.nn.dropout(out, rate=self.dropout)

        return out

    def compute_output_shape(self, input_shape):
        return (input_shape[0], self.units)

    def get_config(self):
        cfg = super(KANLayer, self).get_config()
        cfg.update({
            "units": self.units,
            "grid": self.grid,
            "spline_order": self.spline_order,
            "dropout": self.dropout,
        })
        return cfg
    
# Channel Attention Block
@tf.keras.utils.register_keras_serializable(package="Custom", name="ChannelAttention")
class ChannelAttention(Layer):
    """
    Channel Attention Block (CA) that computes channel-wise attention.
    
    Args:
        reduction: The factor for reducing the number of channels in the intermediate layer.
    """
    def __init__(self, reduction=16, **kwargs):
        super(ChannelAttention, self).__init__(**kwargs)
        self.reduction = reduction

    def build(self, input_shape):
        """
        Build the dense layers for the Channel Attention mechanism.
        - `shared_dense_one`: reduces channel dimensions.
        - `shared_dense_two`: restores the original channel dimensions.
        """
        channel = input_shape[-1]  # Number of channels in the input tensor
        self.shared_dense_one = Dense(channel // self.reduction, activation='relu', kernel_initializer='he_normal', use_bias=True)
        self.shared_dense_two = Dense(channel, kernel_initializer='he_normal', use_bias=True)

    def call(self, inputs):
        """
        Apply the Channel Attention mechanism:
        - Global Average Pooling (avg_pool) and Global Max Pooling (max_pool)
        - Process each through a set of shared dense layers.
        - Combine both attentions and apply a sigmoid function to get the attention weights.
        """
        # Global average and max pooling
        avg_pool = GlobalAveragePooling2D()(inputs)
        max_pool = GlobalMaxPooling2D()(inputs)

        # Apply shared dense layers for both average and max pooled features
        avg_pool = self.shared_dense_one(avg_pool)
        avg_pool = self.shared_dense_two(avg_pool)

        max_pool = self.shared_dense_one(max_pool)
        max_pool = self.shared_dense_two(max_pool)

        # Combine both attention signals and apply sigmoid activation
        attention = Add()([avg_pool, max_pool])
        attention = Activation('sigmoid')(attention)

        # Reshape the attention to match the input dimensions and apply multiplication
        attention = Reshape((1, 1, -1))(attention)
        return Multiply()([inputs, attention])

# Spatial Attention Block
@tf.keras.utils.register_keras_serializable(package="Custom", name="SpatialAttention")
class SpatialAttention(Layer):
    """
    Spatial Attention Block (SA) that computes spatial attention across channels.
    """
    def __init__(self, **kwargs):
        super(SpatialAttention, self).__init__(**kwargs)
        # 2D convolution for spatial attention with a kernel size of 7x7 and sigmoid activation
        self.conv2d = Conv2D(filters=1, kernel_size=7, strides=1, padding='same', activation='sigmoid')

    def call(self, inputs):
        """
        Apply the Spatial Attention mechanism:
        - Apply both average and max pooling across the channel axis to extract spatial features.
        - Concatenate the two features and apply convolution to get spatial attention.
        """
        # Apply global average and max pooling along the channel axis
        avg_pool = tf.reduce_mean(inputs, axis=-1, keepdims=True)
        max_pool = tf.reduce_max(inputs, axis=-1, keepdims=True)

        # Concatenate both pooled features
        concat = Concatenate(axis=-1)([avg_pool, max_pool])

        # Apply convolution to compute spatial attention map
        attention = self.conv2d(concat)

        # Multiply the attention map with the input to highlight important spatial features
        return Multiply()([inputs, attention])


# ----------------------------
# Custom Keras Layer for Res2 Split
# ----------------------------
@register_keras_serializable(package="Custom", name="Res2Split")
class Res2Split(layers.Layer):
    def __init__(self, scale, **kwargs):
        super().__init__(**kwargs)
        self.scale = scale

    def call(self, x):
        channels = tf.shape(x)[-1]
        chunk = channels // self.scale
        splits = []
        for i in range(self.scale):
            splits.append(x[:, :, :, i*chunk:(i+1)*chunk])
        return splits

In [4]:
model = load_model('Proposed Model/Proposed_Model.keras')  
results = full_deployment_profile(
    model,
    input_shape = (224, 224, 3), 
    batch_sizes = (1, 8, 16, 32),
)


  DEPLOYMENT METRICS PROFILE

  [Hardware]
    platform            : macOS-26.3.1-arm64-arm-64bit
    processor           : arm
    python              : 3.11.14
    tensorflow          : 2.16.2
    cpu_cores           : 10
    cpu_threads         : 10
    ram_total_gb        : 16.0
    gpu_devices         : ['/physical_device:GPU:0']
    gpu_name            : METAL

  [Parameters]
    Trainable     :    1,615,025  (1.615 M)
    Non-trainable :        8,192
    Total         :    1,623,217  (1.6232 M)

  [FLOPs]
    Total FLOPs   :   4,286,405,986
    MFLOPs        :         4286.41
    GFLOPs        :          4.2864

  [Memory & Model Size]
    Weight size   :    6.192 MB
    Disk size     :   19.399 MB
    RAM (before)  :   789.77 MB
    RAM (after)   :  1133.84 MB
    RAM delta     :   344.08 MB

  [Latency & Throughput]  (200 timed runs per batch)
   Batch |   Mean ms |  Std ms |   P95 ms |   P99 ms |      img/s
  -------+-----------+---------+----------+----------+-----------
  